In [2]:
!pip install ultralytics
!pip install roboflow pycocotools opencv-python-headless matplotlib tqdm seaborn


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
    --------------------------------------- 0.5/38.8 MB 1.2 MB/s eta 0:00:32
    --------------------------------------- 0.8/38.8 MB 1.2 MB/s eta 0:00:32
   - -------------------------------------- 1.0/38.8 MB 1.2 MB/s eta 0:00:31
   - -------------------------------------- 1.3/38.8 MB 1.2 MB/s eta 0:00:31
   - -------------------------------------- 1.6/38.8 MB 1.2 MB/s eta 0:00:31
   - -------------------------------------- 1.8/


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# project layout

In [8]:
# === Cell 2: project layout (updated for your current folders) ===
from pathlib import Path

ROOT = Path.cwd().parent  # notebook in yolo/training/, project root is yolo/
DATA_DIR = ROOT / "data"
ANNOT_DIR = DATA_DIR / "annotations"

# Actual image folders you showed
TRAIN_IMAGES_DIR = DATA_DIR / "training_data"
TEST_IMAGES_DIR  = DATA_DIR / "test_data"

# Pick annotation filenames with sensible fallbacks
def pick_existing(path_candidates):
    for p in path_candidates:
        if p.exists():
            return p
    return None

ANNOT_TRAIN_COCO = pick_existing([
    ANNOT_DIR / "training_coco.json",
    ANNOT_DIR / "train_coco.json",
    ANNOT_DIR / "coco_train.json",
    ANNOT_DIR / "coco.json"
])

ANNOT_TEST_COCO = pick_existing([
    ANNOT_DIR / "test_coco.json",
    ANNOT_DIR / "testing_coco.json",
    ANNOT_DIR / "coco_test.json",
    ANNOT_DIR / "coco.json"
])

YOLO_LABELS_DIR = DATA_DIR / "labels"
LABELS_TRAIN = YOLO_LABELS_DIR / "train"
LABELS_TEST  = YOLO_LABELS_DIR / "test"

OUTPUT_RUNS = ROOT / "runs"
BASE_WEIGHTS = ROOT / "yolov8x.pt"

print("ROOT:", ROOT)
print("DATA_DIR exists:", DATA_DIR.exists())
print("ANNOT_DIR exists:", ANNOT_DIR.exists())
print("TRAIN_IMAGES_DIR exists:", TRAIN_IMAGES_DIR.exists(), "->", TRAIN_IMAGES_DIR)
print("TEST_IMAGES_DIR exists:", TEST_IMAGES_DIR.exists(), "->", TEST_IMAGES_DIR)
print("Selected TRAIN COCO:", ANNOT_TRAIN_COCO)
print("Selected TEST  COCO:", ANNOT_TEST_COCO)
print("YOLO_LABELS_DIR (will be created):", YOLO_LABELS_DIR)
print("BASE_WEIGHTS exists:", BASE_WEIGHTS.exists())


ROOT: c:\Users\Yash Bhardwaj\Desktop\yolo
DATA_DIR exists: True
ANNOT_DIR exists: True
TRAIN_IMAGES_DIR exists: True -> c:\Users\Yash Bhardwaj\Desktop\yolo\data\training_data
TEST_IMAGES_DIR exists: True -> c:\Users\Yash Bhardwaj\Desktop\yolo\data\test_data
Selected TRAIN COCO: c:\Users\Yash Bhardwaj\Desktop\yolo\data\annotations\training_coco.json
Selected TEST  COCO: c:\Users\Yash Bhardwaj\Desktop\yolo\data\annotations\test_coco.json
YOLO_LABELS_DIR (will be created): c:\Users\Yash Bhardwaj\Desktop\yolo\data\labels
BASE_WEIGHTS exists: True


# coco -> yolo

In [9]:
# === Cell 3: COCO -> YOLO converter for train and test (tolerant filename matching) ===
import json, re
from pathlib import Path
from tqdm import tqdm

def find_disk_image(coco_fname, images_dir: Path):
    """
    Try several strategies to locate the actual image file on disk that corresponds to coco_fname:
    1) exact match
    2) same stem with any extension
    3) any filename that startswith the coco stem
    4) any filename that contains the coco stem
    Returns Path or None.
    """
    coco_fname = str(coco_fname)
    p_exact = images_dir / coco_fname
    if p_exact.exists():
        return p_exact

    stem = Path(coco_fname).stem  # without extension
    # try any extension with same stem
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]:
        p = images_dir / (stem + ext)
        if p.exists():
            return p

    # try files that startwith stem (handles appended hashes)
    for p in images_dir.iterdir():
        if not p.is_file():
            continue
        if p.name.startswith(stem):
            return p

    # try files that contain the stem anywhere
    for p in images_dir.iterdir():
        if not p.is_file():
            continue
        if stem in p.name:
            return p

    return None


def coco_to_yolo_labels(coco_json_path: Path, images_dir: Path, out_labels_dir: Path):
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    with open(coco_json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = {im["id"]: im for im in coco.get("images", [])}
    annotations = coco.get("annotations", [])
    categories = {cat["id"]: cat["name"] for cat in coco.get("categories", [])}

    cat_ids = sorted(categories.keys())
    cat_to_idx = {cid: i for i, cid in enumerate(cat_ids)}

    written = set()
    missing_images = set()
    skipped_annotations = 0

    for ann in tqdm(annotations, desc=f"Converting {coco_json_path.name}"):
        img_id = ann.get("image_id")
        img = images.get(img_id)
        if img is None:
            skipped_annotations += 1
            continue
        file_name = img.get("file_name")
        img_w, img_h = img.get("width"), img.get("height")

        disk_path = find_disk_image(file_name, images_dir)
        if disk_path is None:
            missing_images.add(file_name)
            continue

        bbox = ann.get("bbox", [])
        if len(bbox) != 4:
            skipped_annotations += 1
            continue
        cid = ann.get("category_id")
        if cid not in cat_to_idx:
            skipped_annotations += 1
            continue

        # convert bbox
        x, y, w, h = bbox
        x_c = x + w / 2.0
        y_c = y + h / 2.0
        x_c /= img_w
        y_c /= img_h
        w_rel = w / img_w
        h_rel = h / img_h
        class_idx = cat_to_idx[cid]

        # write label file named after the actual disk image basename
        txt_name = Path(disk_path.name).with_suffix(".txt").name
        out_path = out_labels_dir / txt_name
        with open(out_path, "a", encoding="utf-8") as out:
            out.write(f"{class_idx} {x_c:.6f} {y_c:.6f} {w_rel:.6f} {h_rel:.6f}\n")
        written.add(out_path.name)

    return {
        "labels_written": len(written),
        "missing_images_count": len(missing_images),
        "missing_images_sample": list(missing_images)[:10],
        "skipped_annotations": skipped_annotations,
        "num_annotations": len(annotations),
        "num_images_in_coco": len(images)
    }


# Convert training annotations if available
if ANNOT_TRAIN_COCO and ANNOT_TRAIN_COCO.exists():
    res_train = coco_to_yolo_labels(ANNOT_TRAIN_COCO, TRAIN_IMAGES_DIR, LABELS_TRAIN)
    print("Training labels conversion result:", res_train)
else:
    print("Training COCO not found; skipping training label conversion.")

# Convert test annotations if available
if ANNOT_TEST_COCO and ANNOT_TEST_COCO.exists():
    res_test = coco_to_yolo_labels(ANNOT_TEST_COCO, TEST_IMAGES_DIR, LABELS_TEST)
    print("Test labels conversion result:", res_test)
else:
    print("Test COCO not found; skipping test label conversion.")

# Save names.txt using training categories if available, else try test categories
names_out = DATA_DIR / "names.txt"
if ANNOT_TRAIN_COCO and ANNOT_TRAIN_COCO.exists():
    with open(ANNOT_TRAIN_COCO, "r", encoding="utf-8") as f:
        cats = json.load(f).get("categories", [])
    names = [c["name"] for c in sorted(cats, key=lambda x: x["id"])]
    with open(names_out, "w", encoding="utf-8") as f:
        f.write("\n".join(names))
    print("Wrote class names from training COCO to", names_out)
elif ANNOT_TEST_COCO and ANNOT_TEST_COCO.exists():
    with open(ANNOT_TEST_COCO, "r", encoding="utf-8") as f:
        cats = json.load(f).get("categories", [])
    names = [c["name"] for c in sorted(cats, key=lambda x: x["id"])]
    with open(names_out, "w", encoding="utf-8") as f:
        f.write("\n".join(names))
    print("Wrote class names from test COCO to", names_out)
else:
    print("No COCO files found to write names.txt")


Converting training_coco.json:   3%|▎         | 4363/169174 [09:27<5:57:10,  7.69it/s] 


KeyboardInterrupt: 

# train/val split

In [7]:
from pathlib import Path

def summarize_labels(images_dir: Path, labels_dir: Path):
    images = sorted([p for p in images_dir.glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]])
    labels = sorted([p for p in labels_dir.glob("*.txt")])
    imgs_with_labels = [p for p in images if (labels_dir / p.with_suffix('.txt').name).exists()]
    imgs_without_labels = [p for p in images if not (labels_dir / p.with_suffix('.txt').name).exists()]
    extraneous_labels = [p for p in labels if not (images_dir / p.with_suffix('.jpg').name).exists() and not (images_dir / p.with_suffix('.png').name).exists()]

    return {
        "images_total": len(images),
        "labels_total": len(labels),
        "images_with_labels": len(imgs_with_labels),
        "images_without_labels": len(imgs_without_labels),
        "extraneous_label_files": len(extraneous_labels),
        "examples_missing": imgs_without_labels[:5],
        "examples_extraneous": extraneous_labels[:5]
    }

print("TRAIN summary:")
train_summary = summarize_labels(TRAIN_IMAGES_DIR, LABELS_TRAIN)
print(train_summary)

print("\nTEST summary:")
test_summary = summarize_labels(TEST_IMAGES_DIR, LABELS_TEST)
print(test_summary)

# Quick sanity: sample a few label files
print("\nSample label files (train):", list(LABELS_TRAIN.glob("*.txt"))[:5])
print("Sample label files (test):", list(LABELS_TEST.glob("*.txt"))[:5])


TRAIN summary:
{'images_total': 0, 'labels_total': 0, 'images_with_labels': 0, 'images_without_labels': 0, 'extraneous_label_files': 0, 'examples_missing': [], 'examples_extraneous': []}

TEST summary:
{'images_total': 1085, 'labels_total': 0, 'images_with_labels': 0, 'images_without_labels': 1085, 'extraneous_label_files': 0, 'examples_missing': [WindowsPath('c:/Users/Yash Bhardwaj/Desktop/yolo/data/test_data/video-7cJxWPFMvPdSiWASY-frame-000000-zCbGu9ErmZn6AHfEB.jpg'), WindowsPath('c:/Users/Yash Bhardwaj/Desktop/yolo/data/test_data/video-7cJxWPFMvPdSiWASY-frame-000289-AHgt2W4mdvP4AfaZm.jpg'), WindowsPath('c:/Users/Yash Bhardwaj/Desktop/yolo/data/test_data/video-7cJxWPFMvPdSiWASY-frame-000578-P3mMYp4tbaYXPwZdw.jpg'), WindowsPath('c:/Users/Yash Bhardwaj/Desktop/yolo/data/test_data/video-7cJxWPFMvPdSiWASY-frame-000867-Yd9Bfj6Xy5mD4BsNv.jpg'), WindowsPath('c:/Users/Yash Bhardwaj/Desktop/yolo/data/test_data/video-7cJxWPFMvPdSiWASY-frame-001156-nvQuqtreZrvQ9jBH5.jpg')], 'examples_extraneou

# data.yaml for yolo

In [6]:
# Cell 5: write YOLO-style data YAML
data_yaml = {
    'train': str(IMG_TRAIN.resolve()),
    'val': str(IMG_VAL.resolve()),
    'nc': len(names),
    'names': names
}

import yaml
with open(ROOT / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)
print("Wrote:", ROOT / "data.yaml")


Wrote: c:\Users\Yash Bhardwaj\Desktop\yolo\data.yaml


# finetuning yolov8x

In [ ]:
# Cell 6: Train YOLOv8x using GPU (CUDA)

from ultralytics import YOLO
import torch

# ---- Sanity check (MUST pass for GPU) ----
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# ---- Load pretrained weights ----
model = YOLO(str(ROOT / "yolov8x.pt"))

# ---- Train (fine-tune) ----
model.train(
    data=str(ROOT / "data.yaml"),   # dataset config
    epochs=30,                      # increase if GPU allows
    imgsz=640,
    batch=16,                       # adjust to GPU VRAM (8GB → 8–16)
    device=0,                       # GPU 0 (IMPORTANT)
    project=str(OUTPUT_RUNS),       # runs/
    name="yolov8x_finetune",
    exist_ok=True,
    workers=8,                      # good default for GPU
    amp=True                        # mixed precision (faster, less VRAM)
)

# ---- Output ----
print("Training complete.")
print("Best weights saved at:")
print(OUTPUT_RUNS / "yolov8x_finetune" / "weights" / "best.pt")


NameError: name 'ROOT' is not defined

# tracking distance across frames

In [ ]:
# === Cell 7: Parse Labelbox index.json -> filename -> (videoId, frameIndex) ===
import json
from pathlib import Path

# Choose which index to load (training or test)
ANNOT_VIDEO = ANNOT_DIR / "training_index.json"   # change to test_index.json if needed
IMAGES_DIR = DATA_DIR / "training_data"            # or test_data

with open(ANNOT_VIDEO, "r", encoding="utf-8") as f:
    frames_json = json.load(f)

# Build a lookup of actual filenames on disk
disk_images = list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.png"))
disk_name_map = {p.stem: p.name for p in disk_images}

frame_meta = {}

for fr in frames_json.get("frames", []):
    video_meta = fr.get("videoMetadata", {})
    video_id = video_meta.get("videoId")
    frame_idx = video_meta.get("frameIndex")

    # Labelbox usually stores datasetFrameId which is embedded in filename
    dataset_frame_id = fr.get("datasetFrameId")

    if not dataset_frame_id:
        continue

    # Try to match datasetFrameId to actual disk filenames
    for stem, fname in disk_name_map.items():
        if dataset_frame_id in stem:
            frame_meta[fname] = {
                "videoId": video_id,
                "frameIndex": frame_idx
            }
            break

print("Mapped frame metadata entries:", len(frame_meta))

# Sanity check
sample = list(frame_meta.items())[:5]
for k, v in sample:
    print(k, "->", v)


In [ ]:
# === Cell 8: Tracking + speed estimation (pixel/sec) ===
from ultralytics import YOLO
from collections import defaultdict
from pathlib import Path
import math

# Load trained model
MODEL_PATH = OUTPUT_RUNS / "yolov8x_finetune" / "weights" / "best.pt"
model = YOLO(str(MODEL_PATH))

# Image source (training or test)
SOURCE_DIR = DATA_DIR / "training_data"   # or test_data

DEFAULT_FPS = 30.0  # replace if you have per-video FPS

def center_xyxy(xyxy):
    x1, y1, x2, y2 = xyxy
    return ((x1 + x2) / 2, (y1 + y2) / 2)

# Run tracker
results = model.track(
    source=str(SOURCE_DIR),
    tracker="bytetrack.yaml",
    persist=True,
    stream=True
)

# Collect per-track positions
tracks = defaultdict(list)

for res in results:
    frame_path = Path(res.path).name
    meta = frame_meta.get(frame_path)

    if not meta:
        continue

    frame_idx = meta["frameIndex"]
    t = frame_idx / DEFAULT_FPS

    if res.boxes is None:
        continue

    for box in res.boxes:
        if box.id is None:
            continue

        tid = int(box.id.item())
        xyxy = box.xyxy[0].tolist()
        cx, cy = center_xyxy(xyxy)

        tracks[tid].append((t, cx, cy, frame_path))

# Compute speeds
track_speeds = {}

for tid, pts in tracks.items():
    pts = sorted(pts, key=lambda x: x[0])
    speeds = []

    for (t1, x1, y1, _), (t2, x2, y2, _) in zip(pts, pts[1:]):
        dt = t2 - t1
        if dt <= 0:
            continue

        dist_px = math.hypot(x2 - x1, y2 - y1)
        speed_px_s = dist_px / dt
        speeds.append((t2, speed_px_s))

    track_speeds[tid] = speeds

# Print sample output
for tid, sp in list(track_speeds.items())[:5]:
    print(f"Track {tid} speed samples:", sp[:5])
